In [1]:
import matplotlib.pyplot as plt
from pathlib import Path
import hydra
from tqdm.notebook import tqdm
import pandas as pd
import numpy as np
from omegaconf import OmegaConf
import torch
from lightning.pytorch.loggers import TensorBoardLogger

from svs.modules.datasets import LitNNDataset, xyz_to_zyx, xyzt_to_tzyx, t3xyz_to_t3zyx, t3zyx_to_tzyx3
import svs.modules.transforms as T
from svs.utils.constants import *

In [2]:
transforms = T.Compose([
    T.EnsureFloat(keys=[IMAGE_KEY, MI_KEY, MF_KEY, FWD_FLOW_KEY, BWD_FLOW_KEY]),
    # Reorder axes for tensors
    T.ReorderAxes(keys=[IMAGE_KEY], axes=xyzt_to_tzyx),
    T.ReorderAxes(keys=[MI_KEY, MF_KEY], axes=xyz_to_zyx),
    T.ReorderAxes(keys=[FWD_FLOW_KEY, BWD_FLOW_KEY], axes=t3xyz_to_t3zyx),
    # Add channel dimension
    T.AddDimAt(keys=[MI_KEY, MF_KEY], axis=0),
    T.AddDimAt(keys=[IMAGE_KEY], axis=1),

    # Move flow channel to last dim
    T.ReorderAxes(keys=[FWD_FLOW_KEY, BWD_FLOW_KEY],axes=t3zyx_to_tzyx3)
    
])


data = LitNNDataset(
    num_workers=4,
    batch_size=8,
    train_config={
        'base_dir': '../data/refactor_prep/acdc_lv',
        'imgs_dir': 'NIFTI_4D_Datasets',
        'segs_dir': 'NIFTI_Single_Ventricle_Segmentations',
        'metadata_file': 'Segmentation_volumes.xlsx',
        'split': 'train',
        'forward_flow_subdir': 'optical_flow/forward',
        'backward_flow_subdir': 'optical_flow/backward',
        'transforms': transforms
    },
    val_config={
        'base_dir': '../data/refactor_prep/acdc_lv',
        'imgs_dir': 'NIFTI_4D_Datasets',
        'segs_dir': 'NIFTI_Single_Ventricle_Segmentations',
        'metadata_file': 'Segmentation_volumes.xlsx',
        'split': 'val',
        'forward_flow_subdir': 'optical_flow/forward',
        'backward_flow_subdir': 'optical_flow/backward',
        'transforms': transforms
    }
)

data.setup(stage="fit")

In [3]:
train_loader = data.train_dataloader()
len(train_loader)

10

In [4]:
for batch in train_loader:
    # print(IMAGE_KEY, batch[IMAGE_KEY].shape)
    # print(MI_KEY, batch[MI_KEY].shape, MF_KEY, batch[MF_KEY].shape)
    print(FWD_FLOW_KEY, batch[FWD_FLOW_KEY].shape, BWD_FLOW_KEY, batch[BWD_FLOW_KEY].shape)
    print(batch[OFFSET_KEY])
    print()

fwd_flow torch.Size([8, 12, 80, 80, 80, 3]) bwd_flow torch.Size([8, 12, 80, 80, 80, 3])
tensor([1, 1, 6, 0, 6, 4, 1, 1])

fwd_flow torch.Size([8, 15, 80, 80, 80, 3]) bwd_flow torch.Size([8, 15, 80, 80, 80, 3])
tensor([4, 5, 0, 4, 7, 3, 8, 5])

fwd_flow torch.Size([8, 13, 80, 80, 80, 3]) bwd_flow torch.Size([8, 13, 80, 80, 80, 3])
tensor([0, 3, 0, 6, 5, 5, 8, 4])

fwd_flow torch.Size([8, 11, 80, 80, 80, 3]) bwd_flow torch.Size([8, 11, 80, 80, 80, 3])
tensor([2, 0, 2, 3, 0, 0, 3, 0])

fwd_flow torch.Size([8, 14, 80, 80, 80, 3]) bwd_flow torch.Size([8, 14, 80, 80, 80, 3])
tensor([4, 2, 4, 8, 4, 0, 6, 5])

fwd_flow torch.Size([8, 13, 80, 80, 80, 3]) bwd_flow torch.Size([8, 13, 80, 80, 80, 3])
tensor([0, 4, 6, 4, 2, 6, 4, 2])

fwd_flow torch.Size([8, 14, 80, 80, 80, 3]) bwd_flow torch.Size([8, 14, 80, 80, 80, 3])
tensor([3, 0, 3, 5, 4, 2, 6, 4])

fwd_flow torch.Size([8, 12, 80, 80, 80, 3]) bwd_flow torch.Size([8, 12, 80, 80, 80, 3])
tensor([2, 5, 1, 4, 0, 0, 7, 2])

fwd_flow torch.Size([8, 